In [ ]:
import os
import logging
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==============================================================
# LOGGING
# ==============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

# ==============================================================
# PRETTY PRINTING
# ==============================================================
def banner(title):
    print("\n" + "=" * 70)
    print(f" {title}")
    print("=" * 70)

def kv(key, value):
    print(f"  • {key:<22}: {value}")

# ==============================================================
# CONFIGURATION
# ==============================================================
class InferenceConfig:
    model_path = "/kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x"
    batch_size = 8
    beam_size = 8
    use_fp16 = True
    max_length = 256


# ==============================================================
# INFERENCE ENGINE
# ==============================================================
class UltraInferenceEngine:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._load_model()

    def _load_model(self):
        banner("LOADING MODEL")

        model_dir = os.path.abspath(self.cfg.model_path)

        if not os.path.isdir(model_dir):
            raise RuntimeError(f"❌ Model directory not found:\n{model_dir}")

        kv("Resolved Path", model_dir)
        kv("Device", self.device)

        # Check essential files
        required_files = ["config.json"]
        for f in required_files:
            if not os.path.exists(os.path.join(model_dir, f)):
                raise RuntimeError(f"❌ Missing required file: {f}")

        # ---- LOAD TOKENIZER (LOCAL ONLY) ----
        self.tokenizer = AutoTokenizer.from_pretrained(
            pretrained_model_name_or_path=model_dir,
            local_files_only=True,
            trust_remote_code=False
        )

        # ---- LOAD MODEL (LOCAL ONLY) ----
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            pretrained_model_name_or_path=model_dir,
            local_files_only=True,
            trust_remote_code=False,
            torch_dtype=(
                torch.float16
                if self.cfg.use_fp16 and self.device.type == "cuda"
                else torch.float32
            )
        )

        self.model.to(self.device)
        self.model.eval()

        logger.info("✅ Model successfully loaded from local Kaggle dataset")

    @torch.no_grad()
    def infer(self, texts):
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(self.device)

        outputs = self.model.generate(
            **inputs,
            num_beams=self.cfg.beam_size,
            max_length=self.cfg.max_length
        )

        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)


# ==============================================================
# MAIN
# ==============================================================
def main():
    banner("SYSTEM INFO")
    kv("PyTorch Version", torch.__version__)
    kv("CUDA Available", torch.cuda.is_available())
    kv("Execution Mode", "GPU" if torch.cuda.is_available() else "CPU")

    # ------------------------------------------------------------------
    # 🔹 REPLACE THIS WITH REAL test.csv IF NEEDED
    # ------------------------------------------------------------------
    test_df = pd.DataFrame({
        "id": range(4),
        "text": [
            "𒀀𒈾𒀭𒂗𒆤",
            "𒄿𒈾𒀭𒊩𒌆",
            "𒅆𒀭𒈹𒂗",
            "𒀀𒈾𒀭𒆠"
        ]
    })

    banner("STARTING INFERENCE")
    kv("Samples", len(test_df))

    engine = UltraInferenceEngine(InferenceConfig)
    predictions = engine.infer(test_df["text"].tolist())

    # ==============================================================
    # CREATE SUBMISSION FILE
    # ==============================================================
    submission = pd.DataFrame({
        "id": test_df["id"],
        "prediction": predictions
    })

    output_path = "/kaggle/working/submission.csv"
    submission.to_csv(output_path, index=False)

    banner("SUBMISSION CREATED")
    kv("Saved To", output_path)
    kv("Rows", len(submission))

    print("\nPreview:")
    print(submission.head())


if __name__ == "__main__":
    main()